Inspecting the shape our prediction file 

In [1]:
import pandas as pd

df = pd.read_parquet("data/predictions_2526.parquet")

print("=== Shape ===")
print(df.shape)

print("\n=== Columns ===")
print(df.columns.tolist())

print("\n=== First 3 rows ===")
print(df.head(3).to_string())

=== Shape ===
(29338, 16)

=== Columns ===
['element', 'player_id', 'understat_id', 'gw', 'name', 'position', 'team', 'e_minutes', 'e_points', 'e_points_core', 'exp_bonus', 'pts_goals', 'pts_assists', 'pts_cs', 'pts_dc', 'pts_appear']

=== First 3 rows ===
   element  player_id  understat_id  gw               name position     team  e_minutes  e_points  e_points_core  exp_bonus  pts_goals  pts_assists    pts_cs  pts_dc  pts_appear
0        1          1        9676.0   1  David Raya Martín       GK  Arsenal  57.036650  2.825260       2.641448   0.183812   0.277130     0.144664  0.864091     0.0    1.355562
1        1          1        9676.0   2  David Raya Martín       GK  Arsenal  79.936048  4.611181       4.352299   0.258881   0.675658     0.352699  1.523976     0.0    1.799967
2        1          1        9676.0   3  David Raya Martín       GK  Arsenal  82.790917  3.487996       3.220125   0.267871   0.289221     0.150976  0.917830     0.0    1.862099


Colomns in our file

In [2]:
import pandas as pd

df = pd.read_parquet("data/history/all_seasons_fixed.parquet")

# just the 2025-26 season
df = df[df["season"] == "2025-26"]

print("=== Columns ===")
print(df.columns.tolist())

print("\n=== Price-related columns for one player (element=1), first 3 GWs ===")
cols = [c for c in df.columns if c in ("element", "gw", "round", "value", "name")]
print(df[df["element"] == 1][cols].head(3).to_string())

=== Columns ===
['name', 'assists', 'attempted_passes', 'big_chances_created', 'big_chances_missed', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'completed_passes', 'creativity', 'dribbles', 'ea_index', 'element', 'errors_leading_to_goal', 'errors_leading_to_goal_attempt', 'fixture', 'fouls', 'goals_conceded', 'goals_scored', 'ict_index', 'id', 'influence', 'key_passes', 'kickoff_time', 'kickoff_time_formatted', 'loaned_in', 'loaned_out', 'minutes', 'offside', 'open_play_crosses', 'opponent_team', 'own_goals', 'penalties_conceded', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'round', 'saves', 'selected', 'tackled', 'tackles', 'target_missed', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'winning_goals', 'yellow_cards', 'GW', 'position', 'team', 'season', 'xP', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', '

Making the target that is points and filtering for GW1

In [3]:
import pandas as pd

# our predictions (the value we want to maximize)
preds = pd.read_parquet("data/predictions_2526.parquet")

# historical file that has the price column ("value")
history = pd.read_parquet("data/history/all_seasons_fixed.parquet")
history = history[history["season"] == "2025-26"]

# filter predictions to GW1
gw1 = preds[preds["gw"] == 1].copy()

print("GW1 predictions shape:", gw1.shape)
print("Unique players in GW1:", gw1["element"].nunique())

GW1 predictions shape: (690, 16)
Unique players in GW1: 690


Getting the price of every player to join it to our predictions file

In [4]:
# grab just the price for GW1 from the history file
# round == gameweek in this file
prices = history[history["round"] == 1][["element", "value"]].copy()

print("Price rows for GW1:", prices.shape)
print("Unique players with a price:", prices["element"].nunique())

Price rows for GW1: (692, 2)
Unique players with a price: 690


De Duplicating

In [5]:
# find the elements that appear more than once
dupe_counts = prices["element"].value_counts()
dupes = dupe_counts[dupe_counts > 1]
print("Players with duplicate price rows:", dupes.index.tolist())

# show their actual rows
print("\nTheir rows:")
print(prices[prices["element"].isin(dupes.index)].to_string())

Players with duplicate price rows: [391, 100]

Their rows:
        element  value
224150      391     50
224234      100     45
224434      391     50
224781      100     45


In [6]:
# keep one row per player (they're identical anyway, so "first" is safe)
prices = prices.drop_duplicates(subset="element", keep="first")

print("Price rows after dedupe:", prices.shape)
print("Unique players:", prices["element"].nunique())

Price rows after dedupe: (690, 2)
Unique players: 690


Joining the price to our predictions data

In [7]:
# attach price onto our GW1 predictions
gw1 = gw1.merge(prices, on="element", how="left")

# check the merge worked: any players left without a price?
print("Rows after merge:", gw1.shape)
print("Players missing a price:", gw1["value"].isna().sum())

Rows after merge: (690, 17)
Players missing a price: 0


Colomns required for the optimiser

In [8]:
# the columns the optimizer actually cares about
opt_input = gw1[["element", "name", "position", "team", "value", "e_points"]].copy()

print(opt_input.head(10).to_string())
print("\nPositions present:", opt_input["position"].unique())
print("Number of teams:", opt_input["team"].nunique())

   element                          name position     team  value  e_points
0        1             David Raya Martín       GK  Arsenal     55  2.825260
1        2    Kepa Arrizabalaga Revuelta       GK  Arsenal     45  1.537041
2        3                     Karl Hein       GK  Arsenal     40  0.406434
3        4                 Tommy Setford       GK  Arsenal     40  0.406434
4        5  Gabriel dos Santos Magalhães      DEF  Arsenal     60  3.395044
5        6                William Saliba      DEF  Arsenal     60  3.000388
6        7            Riccardo Calafiori      DEF  Arsenal     55  1.692038
7        8                Jurriën Timber      DEF  Arsenal     55  3.385109
8        9                  Jakub Kiwior      DEF  Arsenal     55  1.574321
9       10            Myles Lewis-Skelly      DEF  Arsenal     55  1.625501

Positions present: <ArrowStringArray>
['GK', 'DEF', 'MID', 'FWD']
Length: 4, dtype: str
Number of teams: 20


In [9]:
print("=== Price sanity (in £m) ===")
print("Cheapest:", opt_input["value"].min() / 10)
print("Most expensive:", opt_input["value"].max() / 10)
print("\n=== Most expensive players (should be the known premiums) ===")
print(opt_input.nlargest(8, "value")[["name", "position", "team", "value"]].to_string())

=== Price sanity (in £m) ===
Cheapest: 4.0
Most expensive: 14.5

=== Most expensive players (should be the known premiums) ===
                       name position         team  value
380           Mohamed Salah      MID    Liverpool    145
429          Erling Haaland      FWD     Man City    140
234             Cole Palmer      MID      Chelsea    105
498          Alexander Isak      FWD    Newcastle    105
15              Bukayo Saka      MID      Arsenal    100
63            Ollie Watkins      FWD  Aston Villa     90
448  Bruno Borges Fernandes      MID      Man Utd     90
665         Viktor Gyökeres      FWD      Arsenal     90


Setting up the Optimiser

In [10]:
import pulp

# reset the index so positions 0..689 line up cleanly with our players
opt_input = opt_input.reset_index(drop=True)

# one binary (0 or 1) decision variable per player
# 1 = "this player is IN the squad", 0 = "not in"
picks = {
    i: pulp.LpVariable(f"pick_{i}", cat="Binary")
    for i in opt_input.index
}

print("Number of decision variables created:", len(picks))
print("Example variable:", picks[0])

Number of decision variables created: 690
Example variable: pick_0


Adding the Objective to the Optimiser

In [11]:
# create the problem, and tell PuLP we're MAXIMIZING
prob = pulp.LpProblem("fpl_squad", pulp.LpMaximize)

# the objective: sum of (switch × that player's expected points)
prob += pulp.lpSum(
    picks[i] * opt_input.loc[i, "e_points"]
    for i in opt_input.index
)

print("Objective added.")
print(prob.objective)

Objective added.
2.825259934965188*pick_0 + 1.5370411565833553*pick_1 + 1.6330187488367125*pick_10 + 1.352068780910021*pick_100 + 0.43862409001776304*pick_101 + 0.43862409001776304*pick_102 + 0.39176427657710405*pick_103 + 0.43862409001776304*pick_104 + 3.31158422017664*pick_105 + 3.3191568927572455*pick_106 + 1.5717901746744662*pick_107 + 0.7991584463944931*pick_108 + 1.2929120153207665*pick_109 + 1.416599964226502*pick_11 + 1.8879176373558122*pick_110 + 1.2635880216703421*pick_111 + 2.664851803586924*pick_112 + 0.4874011698180754*pick_113 + 0.4809508541363185*pick_114 + 0.48961350212897187*pick_115 + 0.47475091687876864*pick_116 + 0.4874011698180754*pick_117 + 3.8256506349634387*pick_118 + 2.405595347991282*pick_119 + 0.5108198687716868*pick_12 + 2.3095906551909584*pick_120 + 1.4118680372898809*pick_121 + 0.8197712342842272*pick_122 + 1.186970002182535*pick_123 + 1.8663350469213515*pick_124 + 1.0098597236549562*pick_125 + 0.7110166027562328*pick_126 + 0.7110166027562328*pick_127 + 0.

Constraint 1: Select only upto 15 players

In [12]:
# exactly 15 players in the squad
prob += pulp.lpSum(picks[i] for i in opt_input.index) == 15

print("Constraint added: squad size = 15")
print("Total constraints so far:", len(prob.constraints))

Constraint added: squad size = 15
Total constraints so far: 1


How many per position are allowed - Constraint 2 

In [13]:
# required count for each position
position_limits = {"GK": 2, "DEF": 5, "MID": 5, "FWD": 3}

for pos, required in position_limits.items():
    # switches for players in THIS position only
    prob += pulp.lpSum(
        picks[i] for i in opt_input.index
        if opt_input.loc[i, "position"] == pos
    ) == required

print("Constraints added: position counts")
print("Total constraints so far:", len(prob.constraints))

Constraints added: position counts
Total constraints so far: 5


Constraint 3: Total price should be under 100 

In [14]:
# total price of selected players must be <= 1000 (£100.0m in tenths)
prob += pulp.lpSum(
    picks[i] * opt_input.loc[i, "value"]
    for i in opt_input.index
) <= 1000

print("Constraint added: budget <= £100m")
print("Total constraints so far:", len(prob.constraints))

Constraint added: budget <= £100m
Total constraints so far: 6


At max only 2 players per team: Constraint 4

In [15]:
# for each team, at most 3 players selected
for team in opt_input["team"].unique():
    prob += pulp.lpSum(
        picks[i] for i in opt_input.index
        if opt_input.loc[i, "team"] == team
    ) <= 3

print("Constraints added: max 3 per club")
print("Total constraints so far:", len(prob.constraints))

Constraints added: max 3 per club
Total constraints so far: 26


In [16]:
# solve it
prob.solve()

# did it find a valid answer?
print("Status:", pulp.LpStatus[prob.status])

Status: Optimal


Best 15 playera including the starting and bench

In [17]:
# pull out the players whose switch is ON (pick == 1)
selected_rows = [
    i for i in opt_input.index
    if picks[i].value() == 1
]

squad = opt_input.loc[selected_rows].copy()

# sort by position then points, nice to read
pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
squad["pos_sort"] = squad["position"].map(pos_order)
squad = squad.sort_values(["pos_sort", "e_points"], ascending=[True, False])

print(squad[["name", "position", "team", "value", "e_points"]].to_string(index=False))

                         name position          team  value  e_points
              Bart Verbruggen       GK      Brighton     45  3.682173
                    Matz Sels       GK Nott'm Forest     50  3.602814
              Virgil van Dijk      DEF     Liverpool     60  4.196784
               Joško Gvardiol      DEF      Man City     60  4.166684
                 Milos Kerkez      DEF     Liverpool     60  4.100432
            Aaron Wan-Bissaka      DEF      West Ham     45  3.508777
           Jan Paul van Hecke      DEF      Brighton     45  3.070490
                Omar Marmoush      MID      Man City     85  4.794165
                   Cody Gakpo      MID     Liverpool     75  4.353817
                 Bryan Mbeumo      MID       Man Utd     80  3.825651
               Enzo Fernández      MID       Chelsea     65  3.515469
              Pape Matar Sarr      MID         Spurs     50  2.391606
               Erling Haaland      FWD      Man City    140  6.291657
João Pedro Junqueira

In [18]:
print("Total spent: £%.1fm" % (squad["value"].sum() / 10))
print("Total expected points:", round(squad["e_points"].sum(), 2))
print("\nPlayers per club:")
print(squad["team"].value_counts().to_string())

Total spent: £100.0m
Total expected points: 58.18

Players per club:
team
Liverpool        3
Man City         3
Chelsea          3
Brighton         2
Nott'm Forest    1
West Ham         1
Man Utd          1
Spurs            1


In [19]:
# helper: find a player's element (ID) by searching their name
def find_player(search):
    matches = opt_input[opt_input["name"].str.contains(search, case=False, na=False)]
    return matches[["element", "name", "position", "team", "value", "e_points"]]

# try it - let's find Salah
find_player("Salah")

,element,name,position,team,value,e_points
380,381,Mohamed Salah,MID,Liverpool,145,5.826314


Function to fix a players and optimise for the rest the 14 players

In [20]:
def optimize_squad(df, locked_elements=None):
    if locked_elements is None:
        locked_elements = []

    prob = pulp.LpProblem("fpl_squad", pulp.LpMaximize)
    picks = {i: pulp.LpVariable(f"pick_{i}", cat="Binary") for i in df.index}

    # objective: maximize total expected points
    prob += pulp.lpSum(picks[i] * df.loc[i, "e_points"] for i in df.index)

    # squad size = 15
    prob += pulp.lpSum(picks[i] for i in df.index) == 15

    # position counts
    for pos, n in {"GK": 2, "DEF": 5, "MID": 5, "FWD": 3}.items():
        prob += pulp.lpSum(picks[i] for i in df.index if df.loc[i, "position"] == pos) == n

    # budget
    prob += pulp.lpSum(picks[i] * df.loc[i, "value"] for i in df.index) <= 1000

    # max 3 per club
    for team in df["team"].unique():
        prob += pulp.lpSum(picks[i] for i in df.index if df.loc[i, "team"] == team) <= 3

    # --- THE NEW PART: lock in chosen players ---
    for elem in locked_elements:
        idx = df.index[df["element"] == elem][0]   # find row for this element
        prob += picks[idx] == 1                     # force switch ON

    prob.solve()
    return prob, picks

Testing the above function 

In [21]:
# lock Salah (element 381) and re-optimize
prob, picks = optimize_squad(opt_input, locked_elements=[381])

print("Status:", pulp.LpStatus[prob.status])

# extract the squad
selected = [i for i in opt_input.index if picks[i].value() == 1]
squad = opt_input.loc[selected].copy()
squad["pos_sort"] = squad["position"].map({"GK":0,"DEF":1,"MID":2,"FWD":3})
squad = squad.sort_values(["pos_sort","e_points"], ascending=[True,False])

print(squad[["name","position","team","value","e_points"]].to_string(index=False))
print("\nTotal spent: £%.1fm" % (squad["value"].sum()/10))
print("Total expected points:", round(squad["e_points"].sum(), 2))
print("Salah in squad:", 381 in squad["element"].values)

Status: Optimal
                         name position          team  value  e_points
              Bart Verbruggen       GK      Brighton     45  3.682173
                    Matz Sels       GK Nott'm Forest     50  3.602814
              Virgil van Dijk      DEF     Liverpool     60  4.196784
               Joško Gvardiol      DEF      Man City     60  4.166684
              Rayan Aït-Nouri      DEF      Man City     60  4.070197
            Aaron Wan-Bissaka      DEF      West Ham     45  3.508777
           Jan Paul van Hecke      DEF      Brighton     45  3.070490
                Mohamed Salah      MID     Liverpool    145  5.826314
                Omar Marmoush      MID      Man City     85  4.794165
                   Cody Gakpo      MID     Liverpool     75  4.353817
               Enzo Fernández      MID       Chelsea     65  3.515469
              Pape Matar Sarr      MID         Spurs     50  2.391606
João Pedro Junqueira de Jesus      FWD       Chelsea     75  3.458224
    

Locking "N" number of players and optimising for the rest of the squad 

In [22]:
def optimize_by_names(df, lock_names=None):
    """Lock any number of players by name, then optimize the rest."""
    if lock_names is None:
        lock_names = []

    # resolve each name to an element ID
    locked_elements = []
    for nm in lock_names:
        matches = df[df["name"].str.contains(nm, case=False, na=False)]
        if len(matches) == 0:
            print(f"⚠️  No player found matching '{nm}' — skipping")
        elif len(matches) > 1:
            print(f"⚠️  '{nm}' matches {len(matches)} players — be more specific:")
            print(matches[["name","team"]].to_string(index=False))
            print("   (skipping this one for now)")
        else:
            locked_elements.append(matches.iloc[0]["element"])
            print(f"🔒 Locked: {matches.iloc[0]['name']}")

    # hand off to the real optimizer
    prob, picks = optimize_squad(df, locked_elements=locked_elements)
    return prob, picks

Testing the above function 

In [23]:
# lock multiple players by name
prob, picks = optimize_by_names(opt_input, lock_names=["Salah", "Haaland", "Bukayo Saka", "O'Reilly", "Virgil"])

print("\nStatus:", pulp.LpStatus[prob.status])

selected = [i for i in opt_input.index if picks[i].value() == 1]
squad = opt_input.loc[selected].copy()
squad["pos_sort"] = squad["position"].map({"GK":0,"DEF":1,"MID":2,"FWD":3})
squad = squad.sort_values(["pos_sort","e_points"], ascending=[True,False])

print(squad[["name","position","team","value","e_points"]].to_string(index=False))
print("\nTotal spent: £%.1fm" % (squad["value"].sum()/10))
print("Total expected points:", round(squad["e_points"].sum(), 2))

🔒 Locked: Mohamed Salah
🔒 Locked: Erling Haaland
🔒 Locked: Bukayo Saka
🔒 Locked: Nico O'Reilly
🔒 Locked: Virgil van Dijk

Status: Optimal
                 name position      team  value  e_points
      Bart Verbruggen       GK  Brighton     45  3.682173
       Mads Hermansen       GK  West Ham     45  3.169719
      Virgil van Dijk      DEF Liverpool     60  4.196784
    Aaron Wan-Bissaka      DEF  West Ham     45  3.508777
   Jan Paul van Hecke      DEF  Brighton     45  3.070490
   Kyle Walker-Peters      DEF  West Ham     45  3.003874
        Nico O'Reilly      DEF  Man City     50  1.788027
        Mohamed Salah      MID Liverpool    145  5.826314
          Bukayo Saka      MID   Arsenal    100  3.206985
Moisés Caicedo Corozo      MID   Chelsea     55  2.901197
     Ryan Gravenberch      MID Liverpool     55  2.869214
      Pape Matar Sarr      MID     Spurs     50  2.391606
       Erling Haaland      FWD  Man City    140  6.291657
           Liam Delap      FWD   Chelsea     65  3

Starting to add more constraints like starting 11 and captain

In [24]:
from optimize import load_gw_data
import pulp

# fresh GW1 data
df = load_gw_data(gw=1)
print("Loaded:", df.shape)
print(df.head(3)[["name","position","team","value","e_points"]].to_string(index=False))

Loaded: (690, 6)
                      name position    team  value  e_points
         David Raya Martín       GK Arsenal     55  2.825260
Kepa Arrizabalaga Revuelta       GK Arsenal     45  1.537041
                 Karl Hein       GK Arsenal     40  0.406434


Adding the three sets for pick (15), start (11), and captain (1)

In [25]:
# three sets of decision variables, one entry per player in each
pick    = {i: pulp.LpVariable(f"pick_{i}",    cat="Binary") for i in df.index}
start   = {i: pulp.LpVariable(f"start_{i}",   cat="Binary") for i in df.index}
captain = {i: pulp.LpVariable(f"captain_{i}", cat="Binary") for i in df.index}

print("pick switches:   ", len(pick))
print("start switches:  ", len(start))
print("captain switches:", len(captain))

pick switches:    690
start switches:   690
captain switches: 690


Adding the three constraints

In [26]:
prob = pulp.LpProblem("fpl_squad_xi", pulp.LpMaximize)

# --- squad-level rules (same as before, all on `pick`) ---

# exactly 15 picked
prob += pulp.lpSum(pick[i] for i in df.index) == 15

# position counts across the 15
for pos, n in {"GK":2, "DEF":5, "MID":5, "FWD":3}.items():
    prob += pulp.lpSum(pick[i] for i in df.index if df.loc[i,"position"]==pos) == n

# budget
prob += pulp.lpSum(pick[i]*df.loc[i,"value"] for i in df.index) <= 1000

# max 3 per club
for team in df["team"].unique():
    prob += pulp.lpSum(pick[i] for i in df.index if df.loc[i,"team"]==team) <= 3

print("Squad rules added. Constraints so far:", len(prob.constraints))

Squad rules added. Constraints so far: 26


Constraint: only 11 to start

In [27]:
# exactly 11 in the starting XI
prob += pulp.lpSum(start[i] for i in df.index) == 11

print("Added: exactly 11 start. Constraints:", len(prob.constraints))

Added: exactly 11 start. Constraints: 27


Start or Bench

In [28]:
# a player can only start if they were picked
for i in df.index:
    prob += start[i] <= pick[i]

print("Added: start ⊆ pick. Constraints:", len(prob.constraints))

Added: start ⊆ pick. Constraints: 717


Constraints for formations

In [29]:
# legal formation for the starting XI
formation = {"GK": (1,1), "DEF": (3,5), "MID": (2,5), "FWD": (1,3)}

for pos, (lo, hi) in formation.items():
    starters_in_pos = pulp.lpSum(
        start[i] for i in df.index if df.loc[i,"position"]==pos
    )
    prob += starters_in_pos >= lo
    prob += starters_in_pos <= hi

print("Added: formation rules. Constraints:", len(prob.constraints))

Added: formation rules. Constraints: 725


Only 1 Captain

In [30]:
# exactly one captain
prob += pulp.lpSum(captain[i] for i in df.index) == 1

print("Added: exactly 1 captain. Constraints:", len(prob.constraints))

Added: exactly 1 captain. Constraints: 726


Captain must be in starting

In [31]:
# captain must be in the starting XI
for i in df.index:
    prob += captain[i] <= start[i]

print("Added: captain ⊆ start. Constraints:", len(prob.constraints))

Added: captain ⊆ start. Constraints: 1416


Setting the Objective

In [32]:
# objective: starting XI points + captain's points AGAIN (the 2x)
prob += (
    pulp.lpSum(start[i]   * df.loc[i,"e_points"] for i in df.index)
    + pulp.lpSum(captain[i] * df.loc[i,"e_points"] for i in df.index)
)

print("Objective set.")

Objective set.


In [33]:
prob.solve(pulp.PULP_CBC_CMD(msg=False))
print("Status:", pulp.LpStatus[prob.status])

Status: Optimal


This optimises for the best starting 11 and trash bench players, as bench players do not score points, too extreme

In [34]:
# pull out picked players and label their role
rows = []
for i in df.index:
    if pick[i].value() == 1:
        if captain[i].value() == 1:
            role = "🅲 CAPTAIN"
        elif start[i].value() == 1:
            role = "start"
        else:
            role = "bench"
        rows.append((df.loc[i,"name"], df.loc[i,"position"], df.loc[i,"team"],
                     df.loc[i,"value"], df.loc[i,"e_points"], role))

team = pd.DataFrame(rows, columns=["name","position","team","value","e_points","role"])

# sort: starters first (by position), then bench
team["role_sort"] = team["role"].map({"🅲 CAPTAIN":0, "start":0, "bench":1})
team["pos_sort"]  = team["position"].map({"GK":0,"DEF":1,"MID":2,"FWD":3})
team = team.sort_values(["role_sort","pos_sort","e_points"], ascending=[True,True,False])

print(team[["name","position","team","value","e_points","role"]].to_string(index=False))

              name position      team  value  e_points      role
   Bart Verbruggen       GK  Brighton     45  3.682173     start
   Virgil van Dijk      DEF Liverpool     60  4.196784     start
    Joško Gvardiol      DEF  Man City     60  4.166684     start
 Aaron Wan-Bissaka      DEF  West Ham     45  3.508777     start
    Jurriën Timber      DEF   Arsenal     55  3.385109     start
Keane Lewis-Potter      DEF Brentford     50  3.319157     start
     Mohamed Salah      MID Liverpool    145  5.826314     start
     Omar Marmoush      MID  Man City     85  4.794165     start
        Cody Gakpo      MID Liverpool     75  4.353817     start
    Enzo Fernández      MID   Chelsea     65  3.515469     start
    Erling Haaland      FWD  Man City    140  6.291657 🅲 CAPTAIN
     Elyh Harrison       GK   Man Utd     40  0.387003     bench
   Romelle Donovan      MID Brentford     45  0.460162     bench
     Ashley Barnes      FWD   Burnley     45  0.458173     bench
       Iwan Morgan      F

In [35]:
starters = team[team["role"]!="bench"]
print("Starters:", len(starters), " Bench:", (team['role']=='bench').sum())
print("Total spent: £%.1fm" % (team["value"].sum()/10))
cap_pts = team[team["role"]=="🅲 CAPTAIN"]["e_points"].iloc[0]
print("Objective (XI pts + captain again): %.2f" % (starters["e_points"].sum() + cap_pts))
print("\nStarting formation:")
print(starters["position"].value_counts().to_string())

Starters: 11  Bench: 4
Total spent: £100.0m
Objective (XI pts + captain again): 53.33

Starting formation:
position
DEF    5
MID    4
GK     1
FWD    1


Adding weight to the bench players to have a decent bench as well

In [36]:
bench_weight = 0.15

# rebuild JUST the objective (rules unchanged)
prob += (
    # 1. starting XI: full points
    pulp.lpSum(start[i] * df.loc[i,"e_points"] for i in df.index)
    # 2. captain: points again (the 2x)
    + pulp.lpSum(captain[i] * df.loc[i,"e_points"] for i in df.index)
    # 3. bench: small fraction of points  (pick - start = "on the bench")
    + bench_weight * pulp.lpSum(
        (pick[i] - start[i]) * df.loc[i,"e_points"] for i in df.index
    )
)

print(f"Objective updated with bench_weight = {bench_weight}")


Objective updated with bench_weight = 0.15


c:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\.venv\Lib\site-packages\pulp\pulp.py:2178: UserWarning: Overwriting previously set objective.
  warnings.warn("Overwriting previously set objective.")


Slightly better bench players with weight for bench = 0.15, but in the main one, we give weight = 0.2 and tag as balanced and then they are two more version, where one is best 15 and one is best 11

In [37]:
prob.solve(pulp.PULP_CBC_CMD(msg=False))
print("Status:", pulp.LpStatus[prob.status])

# same extraction as before
rows = []
for i in df.index:
    if pick[i].value() == 1:
        if captain[i].value() == 1:
            role = "🅲 CAPTAIN"
        elif start[i].value() == 1:
            role = "start"
        else:
            role = "bench"
        rows.append((df.loc[i,"name"], df.loc[i,"position"], df.loc[i,"team"],
                     df.loc[i,"value"], df.loc[i,"e_points"], role))

team = pd.DataFrame(rows, columns=["name","position","team","value","e_points","role"])
team["role_sort"] = team["role"].map({"🅲 CAPTAIN":0, "start":0, "bench":1})
team["pos_sort"]  = team["position"].map({"GK":0,"DEF":1,"MID":2,"FWD":3})
team = team.sort_values(["role_sort","pos_sort","e_points"], ascending=[True,True,False])

print(team[["name","position","team","value","e_points","role"]].to_string(index=False))
print("\nBench total value: £%.1fm" % (team[team['role']=='bench']['value'].sum()/10))

Status: Optimal
              name position        team  value  e_points      role
   Bart Verbruggen       GK    Brighton     45  3.682173     start
   Virgil van Dijk      DEF   Liverpool     60  4.196784     start
    Joško Gvardiol      DEF    Man City     60  4.166684     start
      Milos Kerkez      DEF   Liverpool     60  4.100432     start
 Aaron Wan-Bissaka      DEF    West Ham     45  3.508777     start
     Omar Marmoush      MID    Man City     85  4.794165     start
        Cody Gakpo      MID   Liverpool     75  4.353817     start
       Cole Palmer      MID     Chelsea    105  4.323174     start
      Bryan Mbeumo      MID     Man Utd     80  3.825651     start
    Enzo Fernández      MID     Chelsea     65  3.515469     start
    Erling Haaland      FWD    Man City    140  6.291657 🅲 CAPTAIN
    Mads Hermansen       GK    West Ham     45  3.169719     bench
Jan Paul van Hecke      DEF    Brighton     45  3.070490     bench
     Junior Kroupi      FWD Bournemouth     45

Adding a test to check if the optimiser gives a legal team always

In [38]:
import pandas as pd
import random
from optimize import optimize_squad, POSITION_LIMITS, BUDGET, MAX_PER_CLUB


def make_random_pool(seed):
    """Build one random-but-legal-ish player pool."""
    random.seed(seed)

    positions = {"GK": 6, "DEF": 12, "MID": 12, "FWD": 8}   # plenty per position
    teams = [f"Team{t}" for t in range(10)]                  # 10 clubs

    rows = []
    element = 1
    for pos, count in positions.items():
        for _ in range(count):
            rows.append({
                "element": element,
                "name": f"Player{element}",
                "position": pos,
                "team": random.choice(teams),
                "value": random.randint(40, 140),      # £4.0m–£14.0m
                "e_points": round(random.uniform(0, 10), 2),
            })
            element += 1
    return pd.DataFrame(rows).reset_index(drop=True)


# quick manual check before we bring in Hypothesis
pool = make_random_pool(seed=1)
print("Pool shape:", pool.shape)
print("Per position:")
print(pool["position"].value_counts().to_string())
print("Per team:")
print(pool["team"].value_counts().to_string())

Pool shape: (38, 6)
Per position:
position
DEF    12
MID    12
FWD     8
GK      6
Per team:
team
Team8    7
Team0    6
Team3    5
Team6    5
Team7    4
Team5    4
Team2    2
Team1    2
Team4    2
Team9    1


In [39]:
prob, sol = optimize_squad(pool, mode="balanced")
print("Status:", pulp.LpStatus[prob.status])

Status: Optimal


In [40]:
import importlib
import optimize
importlib.reload(optimize)
from optimize import optimize_squad, MODES

print("MODES available:", MODES)

MODES available: {'best_11': 0.0, 'balanced': 0.2, 'strong_bench': 0.35, 'best_15': 1.0}


In [41]:
from optimize import get_team

def assert_legal_squad(df, sol):
    """Assert the solved team obeys every FPL rule. Raises if not."""
    team = get_team(df, sol)

    # 15 players
    assert len(team) == 15, f"squad size {len(team)} != 15"

    # position counts in the 15
    pos_counts = team["position"].value_counts().to_dict()
    for pos, n in POSITION_LIMITS.items():
        assert pos_counts.get(pos, 0) == n, f"{pos}: {pos_counts.get(pos,0)} != {n}"

    # budget
    spent = team["value"].sum()
    assert spent <= BUDGET, f"spent {spent} > {BUDGET}"

    # max 3 per club
    max_club = team["team"].value_counts().max()
    assert max_club <= MAX_PER_CLUB, f"a club has {max_club} > {MAX_PER_CLUB}"

    # exactly 11 starters
    starters = team[team["role"] != "bench"]
    assert len(starters) == 11, f"{len(starters)} starters != 11"

    # exactly 1 captain, who is a starter
    caps = team[team["role"] == "CAPTAIN"]
    assert len(caps) == 1, f"{len(caps)} captains != 1"

    # legal formation
    fstart = starters["position"].value_counts().to_dict()
    assert fstart.get("GK",0) == 1, "not exactly 1 GK starting"
    assert 3 <= fstart.get("DEF",0) <= 5, "illegal DEF count"
    assert 2 <= fstart.get("MID",0) <= 5, "illegal MID count"
    assert 1 <= fstart.get("FWD",0) <= 3, "illegal FWD count"


# test the checker on the pool we already solved
assert_legal_squad(pool, sol)
print("✓ checker passed on the manual pool")

✓ checker passed on the manual pool


In [42]:
from hypothesis import given, settings, strategies as st

@settings(max_examples=100, deadline=None)   # 100 random pools; no per-test time limit
@given(seed=st.integers(min_value=0, max_value=10_000))
def test_always_legal(seed):
    pool = make_random_pool(seed)
    prob, sol = optimize_squad(pool, mode="balanced")
    # every random pool must solve...
    assert pulp.LpStatus[prob.status] == "Optimal", f"seed {seed}: {pulp.LpStatus[prob.status]}"
    # ...and produce a legal squad
    assert_legal_squad(pool, sol)

# run it
test_always_legal()
print("✓ passed 100 random pools")

AssertionError: seed 54: Infeasible

In [ ]:
bad_pool = make_random_pool(122)
prob, sol = optimize_squad(bad_pool, mode="balanced")
print("Status:", pulp.LpStatus[prob.status])

# what's the cheapest possible legal squad cost?
# (roughly: 2 cheapest GK + 5 cheapest DEF + 5 cheapest MID + 3 cheapest FWD)
cheapest = 0
for pos, n in POSITION_LIMITS.items():
    prices = bad_pool[bad_pool["position"]==pos]["value"].nsmallest(n)
    cheapest += prices.sum()
print("Cheapest possible legal squad: £%.1fm" % (cheapest/10))
print("Budget: £%.1fm" % (BUDGET/10))

Status: Infeasible
Cheapest possible legal squad: £99.5m
Budget: £100.0m


In [ ]:
def make_random_pool(seed):
    """Build a random pool that's always affordable (realistic FPL structure)."""
    random.seed(seed)

    positions = {"GK": 8, "DEF": 20, "MID": 20, "FWD": 12}   # bigger pool
    teams = [f"Team{t}" for t in range(20)]                   # 20 clubs, like real FPL

    rows = []
    element = 1
    for pos, count in positions.items():
        for _ in range(count):
            # skew cheap: most players £4.0-6.0m, a few expensive
            if random.random() < 0.7:
                price = random.randint(40, 60)     # 70% are cheap fodder
            else:
                price = random.randint(60, 140)    # 30% pricier
            rows.append({
                "element": element,
                "name": f"Player{element}",
                "position": pos,
                "team": random.choice(teams),
                "value": price,
                "e_points": round(random.uniform(0, 10), 2),
            })
            element += 1
    return pd.DataFrame(rows).reset_index(drop=True)

# re-check the previously-failing seed
bad_pool = make_random_pool(122)
prob, sol = optimize_squad(bad_pool, mode="balanced")
print("Seed 122 now:", pulp.LpStatus[prob.status])

Seed 122 now: Optimal


In [ ]:
@settings(max_examples=100, deadline=None)
@given(seed=st.integers(min_value=0, max_value=10_000))
def test_always_legal(seed):
    pool = make_random_pool(seed)
    prob, sol = optimize_squad(pool, mode="balanced")
    assert pulp.LpStatus[prob.status] == "Optimal", f"seed {seed}: {pulp.LpStatus[prob.status]}"
    assert_legal_squad(pool, sol)

test_always_legal()
print("✓ passed 100 random pools")

✓ passed 100 random pools


In [ ]:
# right-sized: fewer examples, still thorough
@settings(max_examples=50, deadline=None)
@given(seed=st.integers(min_value=0, max_value=10_000))
def test_all_modes_legal(seed):
    pool = make_random_pool(seed)
    for mode in ["best_11", "balanced", "strong_bench", "best_15"]:
        prob, sol = optimize_squad(pool, mode=mode)
        assert pulp.LpStatus[prob.status] == "Optimal", f"seed {seed}, {mode}: {pulp.LpStatus[prob.status]}"
        assert_legal_squad(pool, sol)

import time
t0 = time.time()
test_all_modes_legal()
print(f"✓ passed 50 pools × 4 modes in {time.time()-t0:.1f}s")

✓ passed 50 pools × 4 modes in 163.5s
